Import simulation outputs from LISFLOOD-FP and package as NetCDF files.

Ensure to specify `voutput` in LISFLOOD-FP parameters file to get velocity as well as water depth.

In [ ]:
import numpy as np
import os
import pandas as pd
import re
import rioxarray as rxr
import shutil
from typing import cast, Literal
import xarray as xr

import graph_creation

# LISFLOOD_OUTPUT_DIR = "/home/aidan/code/data/res_5m_training_data"
# DEM_FILE = "/home/aidan/code/mSWE-GNN/database/raw_datasets_dyce/DEM/DEM_0.xyz"
# POLYGON_FILE = "raw_datasets_dyce/Geometry/dyce_polygon.pol"
# PREFIX = "res_5m_acc_cuda"
# MAX_STEP = 400

# LISFLOOD_OUTPUT_DIR = "/home/aidan/code/data/res_dk15_hydrograph102"
# DEM_FILE = "/home/aidan/code/data/res_dk15_hydrograph102/res_dk15_hydrograph102.dem"
# POLYGON_FILE = "raw_datasets_dyce/Geometry/polygon_0.pol"
# PREFIX = "res_dk15_hydrograph102"
# MAX_STEP = 97

# NAME = "hydrograph_0_highflow"
LISFLOOD_OUTPUT_DIR = f"/home/aidan/code/tmp/"
DEM_FILE = f"/home/aidan/code/mSWE-GNN/database/raw_datasets_dyce/DEM/dyce_lisfloodfp.xyz"
POLYGON_FILE = "/home/aidan/code/mSWE-GNN/database/raw_datasets_dyce/Geometry/dycecut_polygon.pol"
PREFIX = "result_hydrograph_0_highflow"
# MAX_STEP = 1081

LOAD_MESHES_FROM_FILE = True # Set to False to create meshes from polygon file (takes a long time, so set to True to load from file instead)

# Load Meshes
### Generate meshes or load meshes from file

In [ ]:
import pickle

if LOAD_MESHES_FROM_FILE:
    # Beware this will load the mesh with boundary conditions already applied
    with open("raw_datasets_dyce/dycecut_mesh.pkl", "rb") as f:
        meshes = pickle.load(f)
else:
    meshes = graph_creation.create_mesh_dhydro(POLYGON_FILE, 6, False, max_segment_length=500, segmentation_length_type="absolute")
    # meshes = graph_creation.create_mesh_dhydro("raw_datasets_dk15/Geometry/polygon_102.pol", 1, False)
    with open("raw_datasets_dyce/dycecut_mesh.pkl", "wb") as f:
        pickle.dump(meshes, f)

meshes = meshes[4:6]
mesh = meshes[-1] # Highest resolution mesh
meshes

In [ ]:
for mesh in meshes:
    mesh._import_DEM(DEM_FILE)

### Set the boundary condition edges

In [ ]:
# Read boundary condtions
boundary_conditions = pd.read_csv("~/code/flooddata-gen/setup/lisflood-dyce/5m_cut.bci", sep="\s+", names=["boundary_identifier", "coord1", "coord2", "boundary_condition_type", "boundary_condition_value"])

# Interpolate extra points along continuous sections (to ensure no gaps):
INTERPOLATE_POINTS = 50 # Number of points to add between each pair (set to False to disable)
INTERPOLATION_THRESHOLD = 200 # Maximum spacing between two BC points where interpolation will still be applied
if INTERPOLATE_POINTS > 0:
    import scipy.spatial
    tree = scipy.spatial.cKDTree(boundary_conditions[["coord1","coord2"]])
    distances, indices = tree.query(boundary_conditions[["coord1","coord2"]], k=3) # Get the 3 nearest neighbors for each point
    distances = np.vstack([distances[:,[0,1]], distances[:,[0,2]]]) # Stack the distances for the 1st and 2nd nearest neighbors
    indices = np.vstack([indices[:,[0,1]], indices[:,[0,2]]]) # Stack the indices for the 1st and 2nd nearest neighbors
    interpolation_pairs = np.unique(np.sort(indices[distances[:, 1] < INTERPOLATION_THRESHOLD], axis=1), axis=0) # All pairs below the threshold, removing duplicates

    # TODO: This algorithm inserts extra interpolation lines between the last two nodes at the end of each continuous sections. Fix this.

    for start_idx, end_idx in interpolation_pairs:
        extra_rows = pd.concat([boundary_conditions.loc[[start_idx]], pd.Series([np.nan]*INTERPOLATE_POINTS), boundary_conditions.loc[[end_idx]]])
        extra_rows[["boundary_identifier"]] = boundary_conditions["boundary_identifier"][start_idx]
        extra_rows[["boundary_condition_type"]] = boundary_conditions["boundary_condition_type"][start_idx]
        extra_rows[["boundary_condition_value"]] = boundary_conditions["boundary_condition_value"][start_idx]
        extra_rows[["coord1","coord2"]] = extra_rows[["coord1","coord2"]].interpolate(method="linear")
        
        boundary_conditions = pd.concat([boundary_conditions, extra_rows[1:-1]]).reset_index(drop=True)



In [ ]:
boundary_edge_mask = mesh.edge_type == 1 # Mask out non-boundary edges (edges of type 1)

edge_faces = mesh.edge_faces.reshape(-1,2)

# hfix_values = {}

for i, boundary_condition in boundary_conditions.iterrows():
    # Find point source node for input boundary conditions
    pointsource_x, pointsource_y = boundary_condition.coord1, boundary_condition.coord2 # The true pointsource coordinates
    edge_mid_xy = (mesh.node_xy[mesh.edge_index[0]] + mesh.node_xy[mesh.edge_index[1]]) / 2 # Find all edge_midpoints
    dist_to_pointsource = np.ma.array(np.abs(edge_mid_xy[:,0]-pointsource_x) + np.abs(edge_mid_xy[:,1]-pointsource_y), mask=boundary_edge_mask)
    # Find the closest edge to the pointsource
    bc_edge_index = np.argmin(dist_to_pointsource)
    print(f"BC {i}: Found edge {edge_mid_xy[bc_edge_index]}. Distance from true location={np.sqrt(np.sum((edge_mid_xy[bc_edge_index] - np.array([pointsource_x, pointsource_y]))**2)):.03}m")

    # if boundary_condition.boundary_condition_type == "HFIX":
    #     # This if block is based on a faulty premise: mSWE-GNN applies boundary conditions to ghost faces, not the nearest real face
    #     # Set the hfix value for this face to its elevation relative to the underlying DEM
    #     hfix_values[bc_edge_index] = float(boundary_condition.boundary_condition_value) - graph_creation.interpolate_variable([pointsource_x, pointsource_y], mesh.face_xy, mesh.DEM, method="nearest")
    #     hfix_values[bc_edge_index] = max(0, hfix_values[bc_edge_index][0])
    #     print(f"\t Setting HFIX to {hfix_values[bc_edge_index]:.03}m relative to underlying DEM ({float(boundary_condition.boundary_condition_value)}m absolute).")
    #     print(f"DEM value of nearest face: {graph_creation.interpolate_variable([pointsource_x, pointsource_y], mesh.face_xy, mesh.DEM, method='nearest')}")

    # Set edge type to BC edge
    mesh.edge_type[bc_edge_index] = 2

    # Correctly format edge_faces
    if edge_faces[bc_edge_index][0] != -1:
        # If the first -1 (indicating no face on this side) isn't first
        edge_faces[bc_edge_index] = edge_faces[bc_edge_index, ::-1] # Swap the node indices of the edges (graph_creation.Mesh._import_from_map_netcdf relies on this to identify the BC edge)


### Visualise mesh and boundary conditions

In [ ]:
import matplotlib.pyplot as plt
%matplotlib qt
fig, ax = plt.subplots()
graph_creation.plot_faces(mesh, ax)
for bc_edge in mesh.edge_index.T[mesh.edge_type == 2]:
    ax.plot(mesh.node_x[bc_edge], mesh.node_y[bc_edge], color="red")
ax.scatter(boundary_conditions[:27].coord1, boundary_conditions[:27].coord2, color="orange")
ax.scatter(boundary_conditions[27:].coord1, boundary_conditions[27:].coord2, color="yellow")
plt.show()

# Define functions for dataset extraction and generation

In [ ]:
def read_step(step: int, prefix: str, ftype:Literal["wd", "wdfp", "elev", "Vx", "Vy"]|None=None) -> xr.DataArray:
        return cast(xr.DataArray, rxr.open_rasterio(os.path.join(LISFLOOD_OUTPUT_DIR, f"{prefix}-{int(step):04}.{ftype}"), parse_coordinates=True, masked=True))[0]

def extract_parameter(ftype:Literal["wd", "wdfp", "elev", "Vx", "Vy"], max_step, prefix=PREFIX, shape=None):
    results = []
    for step in range(0, max_step+1):
        results.append(read_step(step, prefix, ftype))
    parameter_array = xr.concat(results, "time")
    
    return parameter_array

def extract_parameters(max_step, prefix):
    print("Extracting water depths")
    wd = extract_parameter("wd", max_step, prefix=prefix)
    print("Extracting x velocities")
    Vx = extract_parameter("Vx", max_step, shape=wd.shape[1:], prefix=prefix)
    print("Extracting y velocities")
    Vy = extract_parameter("Vy", max_step, shape=wd.shape[1:], prefix=prefix)
    return wd, Vx, Vy

In [ ]:
face_coords = xr.Dataset(
    coords={
        "mesh2d_nFaces": ("mesh2d_nFaces", range(mesh.face_xy.shape[0])),
        "x": ("mesh2d_nFaces", mesh.face_xy[:, 0]),
        "y": ("mesh2d_nFaces", mesh.face_xy[:, 1]),
    }
)

def generate_dataset(mesh, hydrograph_name=None, wd=None, Vx=None, Vy=None, save_output=False):
    # The +1 on variables which are indicies is expected by graph_creation on loading in the NetCDF file.

    mesh_dataset = {
        "mesh2d_node_x": xr.DataArray(mesh.node_x, dims=["mesh2d_nNodes"]),
        "mesh2d_node_y": xr.DataArray(mesh.node_y, dims=["mesh2d_nNodes"]),
        "mesh2d_face_x": xr.DataArray(mesh.face_x, dims=["mesh2d_nFaces"]),
        "mesh2d_face_y": xr.DataArray(mesh.face_y, dims=["mesh2d_nFaces"]),
        "mesh2d_edge_nodes": xr.DataArray(mesh.edge_index.T + 1, dims=["mesh2d_nEdges", "Two"]),
        
        "mesh2d_edge_type": xr.DataArray(mesh.edge_type, dims=["mesh2d_nEdges"]),
        "mesh2d_edge_faces": xr.DataArray(edge_faces + 1, dims=["mesh2d_nEdges", "Two"]),
        "mesh2d_face_nodes": xr.DataArray(graph_creation.get_face_nodes_mesh(mesh) + 1, dims=["mesh2d_face_nodes", "mesh2d_nMax_face_nodes"]),
        
        "mesh2d_dem": xr.DataArray(mesh.DEM, dims=["mesh2d_nFaces"]),
    }

    if wd is not None and Vx is not None and Vy is not None:
        mesh_dataset.update({
            "mesh2d_waterdepth": wd.sel(x=face_coords["x"], y=face_coords["y"], method="nearest").reset_coords(drop=True),
            "mesh2d_ucx": Vx.sel(x=face_coords["x"], y=face_coords["y"], method="nearest").reset_coords(drop=True),
            "mesh2d_ucy": Vy.sel(x=face_coords["x"], y=face_coords["y"], method="nearest").reset_coords(drop=True),
        })

    simulation_output = xr.Dataset(mesh_dataset)

    if save_output:
        simulation_output.reset_index("mesh2d_nFaces").to_netcdf(f"{hydrograph_name}.nc", format="NETCDF4")
    return simulation_output

# Compressed NetCDF output
Use the following section to generate a compressed NetCDF datasets for a folder of LISFLOOD-FP outputs

In [ ]:
SOURCE_FOLDER = "/one/exageo/data/dyce-lisflood-fp-cpu-training-data/"
DEST_FOLDER = "/home/aidan/code/mSWE-GNN/database/raw_datasets_dyce/Simulations_6meshes"
TMP_FOLDER = "/home/aidan/code/tmp/"

In [ ]:
completed_hydrographs = set([file.rstrip(".nc.zst") for file in os.listdir(DEST_FOLDER) if file.startswith("hydrograph") and file.endswith(".nc.zst")])

In [ ]:
todo = set([file.rstrip(".tar.zst") for file in os.listdir(SOURCE_FOLDER) if file.endswith(".tar.zst")]) - completed_hydrographs

while todo:
    hydrograph_name = min(todo)
    completed_hydrographs.add(hydrograph_name)
    todo = set([file.rstrip(".tar.zst") for file in os.listdir(SOURCE_FOLDER) if file.endswith(".tar.zst")]) - completed_hydrographs
    with open("broken_hydrographs.txt") as f:
        if hydrograph_name in f.read():
            print(f"Skipping hydrograph {hydrograph_name} as it is marked as broken in broken_hydrographs.txt")
            continue
    
    print(f"Processing hydrograph {hydrograph_name}")
    print(f"Copying and extracting {hydrograph_name}.tar.zst to {TMP_FOLDER}")
    os.system(f"tar --zstd -xvf {os.path.join(SOURCE_FOLDER, f'{hydrograph_name}.tar.zst')} -C {TMP_FOLDER} >/dev/null")
    
    max_step = re.findall(r'.*-(\d\d\d\d)\.wd', "\n".join(os.listdir(os.path.join(TMP_FOLDER,f"result_{hydrograph_name}"))))
    max_step = int(sorted(max_step)[-1])
    max_step_expected = int(int(os.popen(f"tail -n 1 /home/aidan/code/flooddata-gen/data/dyce_hydrographs/{hydrograph_name}.bdy | awk '{{print $2}}'").read()) / 180)
    print(f"Expected final timestep: {max_step_expected}. Actual final timestep: {max_step}")
    if max_step != max_step_expected:
        print(f"WARNING: Expected final timestep {max_step_expected} does not match actual final timestep {max_step} for hydrograph {hydrograph_name}. Skipping this hydrograph.")
        os.system(f"echo WARNING: Expected final timestep {max_step_expected} does not match actual final timestep {max_step} for hydrograph {hydrograph_name}. Skipping this hydrograph. >> broken_hydrographs.txt")
        print(f"Cleaning up extracted folder")
        os.system(f"rm -r {os.path.join(TMP_FOLDER, f'result_{hydrograph_name}')}")
        print("Done\n")
        continue
    
    print(f"Extracting parameters for {hydrograph_name} ({max_step+1} timesteps)")
    try:
        wd, Vx, Vy = extract_parameters(max_step, prefix=f"result_{hydrograph_name}/result_{hydrograph_name}")
    except Exception as e:
        print(f"WARNING: hydrograph {hydrograph_name} gave error {e}. Skipping this hydrograph.")
        os.system(f"echo WWARNING: hydrograph {hydrograph_name} gave error {e}. Skipping this hydrograph. >> broken_hydrographs.txt")
        print(f"Cleaning up extracted folder")
        os.system(f"rm -r {os.path.join(TMP_FOLDER, f'result_{hydrograph_name}')}")
        print("Done\n")
        continue

    print(f"Generating dataset for {hydrograph_name}")
    generate_dataset(mesh, hydrograph_name, wd, Vx, Vy, save_output=True)
    
    print(f"Compressing dataset {hydrograph_name}.nc")
    os.system(f"zstd -10 {hydrograph_name}.nc")
    os.rename(f"{hydrograph_name}.nc.zst", os.path.join(DEST_FOLDER, f"{hydrograph_name}.nc.zst"))
    print(f"Cleaning up extracted folder and uncompressed .nc file")
    os.system(f"rm -r {os.path.join(TMP_FOLDER, f'result_{hydrograph_name}')}")
    os.system(f"rm {hydrograph_name}.nc")
    print("Done.\n")


## Zarr Template Mesh Output
Generate a template mesh (with no wd, Vx, Vy flow data), creating a zarr dataset.

In [ ]:
out_mesh_name = "mesh_2meshes_cut_withBC"

In [ ]:
if os.path.exists(f"{out_mesh_name}.zarr"):
    print(f"Removing existing {out_mesh_name}.zarr")
    shutil.rmtree(f"{out_mesh_name}.zarr")

In [ ]:
import xarray as xr
import zarr

ds = generate_dataset(meshes[-1])

# flowvars_encoding = {"compressor":zarr.codecs.Zstd(level=19), "scale_factor":0.001, "dtype":"int16", "filters":[zarr.codecs.Delta(dtype="int16")]}
coordinate_encoding = {"compressor":zarr.codecs.Zstd(level=19), "dtype": "float64"}
elevation_encoding = coordinate_encoding
index_encoding = {"compressor":zarr.codecs.Zstd(level=19), "dtype": "int32"}
smallint_encoding = {"compressor":zarr.codecs.Zstd(level=19), "dtype": "int8"}

# ds_chunked = ds.chunk({"time":64}) # Time chunks of 64 yields chunksizes of ~1MB. Aiming for the largest chunksize where reads (from a network drive) will still be seek-limited (the application favours small chunk sizes for random access)

ds.to_zarr(
    f"{out_mesh_name}.zarr",
    encoding={
        "mesh2d_node_x": coordinate_encoding,
        "mesh2d_node_y": coordinate_encoding,
        "mesh2d_face_x": coordinate_encoding,
        "mesh2d_face_y": coordinate_encoding,
        "mesh2d_edge_nodes": index_encoding,
        "mesh2d_edge_type": smallint_encoding,
        "mesh2d_edge_faces": index_encoding,
        # "mesh2d_waterdepth": flowvars_encoding,
        # "mesh2d_ucx": flowvars_encoding,
        # "mesh2d_ucy": flowvars_encoding,
        "mesh2d_dem": elevation_encoding
    }
)